# Manipulation Technique Classifier - Legal-BERT

Dialogue -> Legal-Bert encoder -> Multi-label classifier -> 11 manipulation techniques

## 1. Install & Import

In [1]:
!pip install transformers datasets scikit-learn torch accelerate optuna -q

In [2]:
import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import Counter
import optuna
from optuna.samplers import TPESampler

In [3]:
warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Config

In [ ]:
TRAIN_INTENT_CSV = "/kaggle/input/datasets/linhtron123/train-intent/train_intent.csv"
TRAIN_LABEL_CSV  = "/kaggle/input/datasets/linhtron123/legaldata/train_split.csv"
VAL_INTENT_CSV   = "/kaggle/input/datasets/linhtron123/val-intent/val_intent.csv"
VAL_LABEL_CSV    = "/kaggle/input/datasets/linhtron123/legaldata/val_split.csv"
 
OUTPUT_DIR = "/kaggle/working/bert_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
BASE_MODEL     = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH     = 512
STRIDE         = 256
SEED           = 42
N_TRIALS       = 10
EPOCHS_HPO     = 6
PATIENCE_HPO   = 3
EPOCHS_FINAL   = 30
PATIENCE_FINAL = 7

# MaxPoolBERT: k=3 tốt nhất trên small datasets (Table 6, Behrendt et al. 2025)
K_LAYERS = 3
 
COL_DIALOGUE   = "Dialogue"
COL_TECHNIQUES = "Manipulation Techniques"
COL_PLAINTIFF  = "PLAINTIFF"
COL_DEFENDANT  = "DEFENDANT"
 
TECHNIQUES = [
    "persuasion", "playing the victim", "gaslighting", "evasion",
    "deflection", "minimization", "dismissal", "guilt tripping",
    "emotional appeal", "framing the narrative", "character attack"
]
NUM_LABELS = len(TECHNIQUES)
 
torch.manual_seed(SEED)
np.random.seed(SEED)

## 3. Load & Prep Data

In [5]:
def parse_techniques(tech_str):
    if pd.isna(tech_str) or str(tech_str).strip().lower() in ['none', 'nan', '']:
        return []
    return [t.strip().lower() for t in str(tech_str).split(',')
            if t.strip().lower() in TECHNIQUES]
 
def normalize_dialogue(text):
    if pd.isna(text): return ""
    return str(text).strip().replace('\r\n', '\n').replace('\r', '\n')
 
def load_and_merge(intent_csv, label_csv):
    intent_df = pd.read_csv(intent_csv)
    label_df  = pd.read_csv(label_csv)
    intent_df["Dialogue"] = intent_df["Dialogue"].apply(normalize_dialogue)
    label_df["Dialogue"]  = label_df["Dialogue"].apply(normalize_dialogue)
    merged = intent_df.merge(label_df[["Dialogue", COL_TECHNIQUES]],
                             on="Dialogue", how="left")
    merged['techniques_list'] = merged[COL_TECHNIQUES].apply(parse_techniques)
    return merged
 
print("Loading data...")
train_df = load_and_merge(TRAIN_INTENT_CSV, TRAIN_LABEL_CSV)
val_df   = load_and_merge(VAL_INTENT_CSV,   VAL_LABEL_CSV)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")
 
mlb = MultiLabelBinarizer(classes=TECHNIQUES)
mlb.fit([TECHNIQUES])
 
def build_input_text(row):
    dialogue  = str(row[COL_DIALOGUE]).strip()
    plaintiff = str(row[COL_PLAINTIFF]).strip() if pd.notna(row.get(COL_PLAINTIFF)) else ""
    defendant = str(row[COL_DEFENDANT]).strip() if pd.notna(row.get(COL_DEFENDANT)) else ""
    if plaintiff and defendant:
        return f"[PLAINTIFF INTENT] {plaintiff} [DEFENDANT INTENT] {defendant} [DIALOGUE] {dialogue}"
    return dialogue
 
train_texts  = [build_input_text(r) for _, r in train_df.iterrows()]
train_labels = mlb.transform(train_df['techniques_list'])
val_texts    = [build_input_text(r) for _, r in val_df.iterrows()]
val_labels   = mlb.transform(val_df['techniques_list'])
 
print(f"Train: {len(train_texts)} | Val: {len(val_texts)}")
print(f"Label shape: {train_labels.shape}")
 
print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

Loading data...
Train: 722 | Val: 155
Train: 722 | Val: 155
Label shape: (722, 11)
Loading tokenizer: nlpaueb/legal-bert-base-uncased


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## 4. Dataset với Sliding Window

In [6]:
def encode_with_sliding_window(texts, tokenizer, max_length=512, stride=256):
    all_input_ids, all_attn_masks, chunk_to_sample = [], [], []
    for sample_idx, text in enumerate(texts):
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(tokens) <= max_length - 2:
            enc = tokenizer(text, max_length=max_length, padding='max_length',
                            truncation=True, return_tensors='pt')
            all_input_ids.append(enc['input_ids'][0])
            all_attn_masks.append(enc['attention_mask'][0])
            chunk_to_sample.append(sample_idx)
        else:
            for start in range(0, len(tokens), stride):
                end   = min(start + max_length - 2, len(tokens))
                chunk = [tokenizer.cls_token_id] + tokens[start:end] + [tokenizer.sep_token_id]
                pad   = max_length - len(chunk)
                mask  = [1] * len(chunk) + [0] * pad
                chunk = chunk + [tokenizer.pad_token_id] * pad
                all_input_ids.append(torch.tensor(chunk))
                all_attn_masks.append(torch.tensor(mask))
                chunk_to_sample.append(sample_idx)
                if end == len(tokens):
                    break
    return (torch.stack(all_input_ids), torch.stack(all_attn_masks), chunk_to_sample)
 
class ManipulationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512, stride=256):
        input_ids, attn_masks, self.chunk_to_sample = encode_with_sliding_window(
            texts, tokenizer, max_length, stride)
        self.input_ids  = input_ids
        self.attn_masks = attn_masks
        self.labels     = torch.FloatTensor(labels)
 
    def __len__(self):
        return len(self.input_ids)
 
    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attn_masks[idx],
            'labels':         self.labels[self.chunk_to_sample[idx]]
        }

print("Building datasets...")
train_dataset = ManipulationDataset(train_texts, train_labels, tokenizer, MAX_LENGTH, STRIDE)
val_dataset   = ManipulationDataset(val_texts,   val_labels,   tokenizer, MAX_LENGTH, STRIDE)
print(f"Train chunks: {len(train_dataset)} | Val chunks: {len(val_dataset)}")

Token indices sequence length is longer than the specified maximum sequence length for this model (1907 > 512). Running this sequence through the model will result in indexing errors


Building datasets...
Train chunks: 3800 | Val chunks: 822


## 5. Loss: Asymmetric Focal Loss

In [7]:
class AsymmetricFocalLoss(nn.Module):
    """
    Args:
        gamma_pos (float): focusing parameter cho positive examples.
                           Thường để 0 hoặc 1 (Huang et al. default: 0).
        gamma_neg (float): focusing parameter cho negative examples.
                           Thường lớn hơn gamma_pos (default: 4).
        clip (float):      probability margin để shift negative probs.
                           Ngăn model quá tự tin với negative (default: 0.05).
        eps (float):       numerical stability cho log.
    """
    def __init__(self, gamma_pos: float = 0, gamma_neg: float = 4,
                 clip: float = 0.05, eps: float = 1e-8):
        super().__init__()
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.clip      = clip
        self.eps       = eps
 
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs     = torch.sigmoid(logits)
 
        # Clip negative probabilities (Huang et al. Eq. 3)
        probs_neg = (1.0 - probs).clamp(min=self.clip)
 
        # Binary cross-entropy terms
        loss_pos = targets       * torch.log(probs.clamp(min=self.eps))
        loss_neg = (1 - targets) * torch.log(probs_neg.clamp(min=self.eps))
 
        # Asymmetric focusing weights
        # Positive: (1 - p)^gamma_pos  — thường gamma_pos=0 → weight=1
        # Negative: (p_shifted)^gamma_neg — down-weight easy negatives mạnh
        loss_pos = loss_pos * (1.0 - probs)          ** self.gamma_pos
        loss_neg = loss_neg * (1.0 - probs_neg)      ** self.gamma_neg
 
        loss = -(loss_pos + loss_neg)
        return loss.mean()


## 6. Model: Max_CLS Pooling

In [8]:
class ManipulationClassifier(nn.Module):
    """ 
    Pooling strategy:
      Max_CLS: lấy [CLS] token từ k layers cuối,
      element-wise max-pool theo chiều layer depth.
    """
    def __init__(self, model_name: str, num_labels: int,
                 dropout: float = 0.3, k_layers: int = K_LAYERS,
                 gamma_pos: float = 0, gamma_neg: float = 4,
                 clip: float = 0.05):
        super().__init__()
        self.k_layers = k_layers
 
        self.encoder = AutoModel.from_pretrained(
            model_name,
            output_hidden_states=True
        )
        hidden_size = self.encoder.config.hidden_size
 
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.LayerNorm(hidden_size // 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_labels)
        )
 
        self.loss_fct = AsymmetricFocalLoss(
            gamma_pos=gamma_pos,
            gamma_neg=gamma_neg,
            clip=clip
        )
 
    def max_cls_pool(self, hidden_states: tuple) -> torch.Tensor:
        """
        hidden_states: tuple của (num_layers+1) tensors, mỗi tensor (B, T, H)
            - Index 0: embedding layer (không dùng)
            - Index 1..12: output của 12 BERT layers
        Returns:
            (B, H) — max-pooled [CLS] representation
        """
        # Lấy [CLS] (token index 0) từ k layers cuối
        # hidden_states[-1] = layer 12, [-2] = layer 11, ..., [-k] = layer (13-k)
        cls_per_layer = torch.stack(
            [hidden_states[-(i + 1)][:, 0, :] for i in range(self.k_layers)],
            dim=1
        )  # shape: (B, k, H)
 
        # Element-wise max qua k layers → (B, H)
        # torch.max trả về (values, indices); chỉ lấy values
        return cls_per_layer.max(dim=1).values
 
    def forward(self, input_ids, attention_mask, labels=None):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.max_cls_pool(out.hidden_states)   # (B, H)
        logits = self.classifier(pooled)                # (B, num_labels)
 
        loss = None
        if labels is not None:
            loss = self.loss_fct(logits, labels.to(logits.device))
        return loss, logits
 
    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False
 
    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True
 

## 7. Helper

In [9]:
def get_all_preds(model, loader, chunk_to_sample, n_samples):
    """
    Aggregate chunk-level predictions về sample-level.
    Dùng max aggregation qua chunks (consistent với Max_CLS spirit): một chunk chứa evidence mạnh là đủ để trigger nhãn.
    """
    model.eval()
    all_probs  = [[] for _ in range(n_samples)]
    all_labels = [None] * n_samples
    chunk_idx  = 0
    with torch.no_grad():
        for batch in loader:
            iids   = batch['input_ids'].to(DEVICE)
            amask  = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].numpy()
            _, logits = model(iids, amask)
            probs = torch.sigmoid(logits).cpu().numpy()
            for i in range(len(probs)):
                s = chunk_to_sample[chunk_idx]
                all_probs[s].append(probs[i])
                all_labels[s] = labels[i]
                chunk_idx += 1
    # Max aggregation qua chunks
    final_probs  = np.array([np.max(p, axis=0) for p in all_probs])
    final_labels = np.array(all_labels)
    return final_probs, final_labels
 
def evaluate_macro_f1(model, loader, chunk_to_sample, n_samples, threshold=0.5):
    probs, labels = get_all_preds(model, loader, chunk_to_sample, n_samples)
    preds = (probs >= threshold).astype(int)
    return f1_score(labels, preds, average='macro', zero_division=0), probs, labels

## 8. Threshold Tuning

In [10]:
def tune_thresholds(probs, labels):
    """
    Per-class threshold tuning tối ưu F1 trực tiếp trên validation set.
    """
    best_thresholds = np.zeros(NUM_LABELS)

    print(f"\n{'Technique':<25} {'Thresh':>8} {'F1':>8} {'P':>8} {'R':>8}")
    print("-" * 55)

    for i, tech in enumerate(TECHNIQUES):
        best_f1, best_thresh = 0.0, 0.5

        for thresh in np.arange(0.05, 0.95, 0.025):
            preds = (probs[:, i] >= thresh).astype(int)
            f1    = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thresh = f1, thresh

        best_thresholds[i] = best_thresh

        preds_best = (probs[:, i] >= best_thresh).astype(int)
        p = precision_score(labels[:, i], preds_best, zero_division=0)
        r = recall_score(labels[:, i], preds_best, zero_division=0)
        print(f"  {tech:<23} {best_thresh:>8.3f} {best_f1:>8.3f} {p:>8.3f} {r:>8.3f}")

    return best_thresholds

## 9. Objective Function

In [11]:
def objective(trial: optuna.Trial) -> float:
    # Hyperparameters gốc
    lr_encoder    = trial.suggest_float("lr_encoder",   1e-6, 5e-5, log=True)
    lr_head       = trial.suggest_float("lr_head",      1e-4, 5e-3, log=True)
    dropout       = trial.suggest_float("dropout",      0.1,  0.5,  step=0.05)
    weight_decay  = trial.suggest_float("weight_decay", 0.0,  0.1,  step=0.01)
    batch_size    = trial.suggest_categorical("batch_size",  [4, 8, 16])
    grad_accum    = trial.suggest_categorical("grad_accum",  [2, 4, 8])
    freeze_epochs = trial.suggest_int("freeze_epochs",  0, 4)
 
    # AFL hyperparameters (Huang et al., ICCV 2021)
    # gamma_neg: 2-6 theo paper; gamma_pos: 0 hoặc 1
    gamma_neg = trial.suggest_int("gamma_neg",   2, 6)
    gamma_pos = trial.suggest_int("gamma_pos",   0, 1)
    clip      = trial.suggest_float("clip",      0.0, 0.1, step=0.025)
 
    print(f"\n{'='*65}")
    print(f"TRIAL {trial.number:02d} | "
          f"lr_enc={lr_encoder:.1e} | lr_head={lr_head:.1e} | "
          f"dropout={dropout} | wd={weight_decay}")
    print(f"         bs={batch_size} | accum={grad_accum} | "
          f"freeze={freeze_epochs} | "
          f"γ_neg={gamma_neg} | γ_pos={gamma_pos} | clip={clip:.3f}")
    print(f"{'='*65}")
 
    trial_train_loader = DataLoader(train_dataset, batch_size=batch_size,
                                    shuffle=True, num_workers=2)
    trial_val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                                    shuffle=False, num_workers=2)
 
    model = ManipulationClassifier(
        BASE_MODEL, NUM_LABELS,
        dropout=dropout,
        k_layers=K_LAYERS,
        gamma_pos=gamma_pos,
        gamma_neg=gamma_neg,
        clip=clip
    ).to(DEVICE)
 
    if freeze_epochs > 0:
        model.freeze_encoder()
        optimizer = AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr_head, weight_decay=weight_decay
        )
    else:
        optimizer = AdamW([
            {'params': model.encoder.parameters(),    'lr': lr_encoder, 'weight_decay': weight_decay},
            {'params': model.classifier.parameters(), 'lr': lr_head,    'weight_decay': weight_decay}
        ])
 
    total_steps = len(trial_train_loader) * EPOCHS_HPO
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=total_steps // 10,
        num_training_steps=total_steps
    )
 
    best_f1      = 0.0
    patience_cnt = 0
 
    for epoch in range(EPOCHS_HPO):
        if epoch == freeze_epochs and freeze_epochs > 0:
            model.unfreeze_encoder()
            optimizer = AdamW([
                {'params': model.encoder.parameters(),    'lr': lr_encoder, 'weight_decay': weight_decay},
                {'params': model.classifier.parameters(), 'lr': lr_head,    'weight_decay': weight_decay}
            ])
            remaining = EPOCHS_HPO - freeze_epochs
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=len(trial_train_loader) * remaining // 10,
                num_training_steps=len(trial_train_loader) * remaining
            )
            patience_cnt = 0
 
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
 
        for step, batch in enumerate(trial_train_loader):
            iids   = batch['input_ids'].to(DEVICE)
            amask  = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
 
            loss, _ = model(iids, amask, labels)
            (loss / grad_accum).backward()
 
            if (step + 1) % grad_accum == 0 or (step + 1) == len(trial_train_loader):
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
 
            total_loss += loss.item()
 
        avg_loss = total_loss / len(trial_train_loader)
        macro_f1, _, _ = evaluate_macro_f1(
            model, trial_val_loader,
            val_dataset.chunk_to_sample, len(val_texts)
        )
 
        phase = "[FROZEN]" if epoch < freeze_epochs else "[FULL  ]"
        print(f"  Epoch {epoch+1:>2}/{EPOCHS_HPO} {phase} | "
              f"Loss: {avg_loss:.4f} | Macro F1: {macro_f1:.4f}")
 
        trial.report(macro_f1, epoch)
        if trial.should_prune():
            print(f"  → Pruned at epoch {epoch+1}")
            del model; gc.collect(); torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()
 
        if macro_f1 > best_f1:
            best_f1      = macro_f1
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE_HPO:
                print(f"  → Early stop (patience={PATIENCE_HPO})")
                break
 
    print(f"  Trial {trial.number:02d} best Macro F1 = {best_f1:.4f}")
    del model; gc.collect()
    try: torch.cuda.empty_cache()
    except Exception: pass
    return best_f1
 

## 10. Optuna Study

In [12]:
print(f"\nOptuna HPO: {N_TRIALS} trials | Metric: Val Macro F1 (maximize)\n")
 
sampler = TPESampler(seed=SEED)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
 
study = optuna.create_study(
    direction="maximize",
    study_name="legalbert-v2-hpo",
    sampler=sampler,
    pruner=pruner,
)
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
    catch=(Exception,),
)
 
best = study.best_trial
print("\n" + "=" * 65)
print("BEST TRIAL")
print("=" * 65)
print(f"Best Macro F1: {best.value:.4f}")
print("Hyperparameters:")
for k, v in best.params.items():
    print(f"  {k}: {v}")
 
results = {
    "best_macro_f1": best.value,
    "best_params": best.params,
    "all_trials": [
        {"trial": t.number, "macro_f1": t.value,
         "params": t.params, "state": str(t.state)}
        for t in study.trials
    ]
}
with open(f"{OUTPUT_DIR}/optuna_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved: {OUTPUT_DIR}/optuna_results.json")

[I 2026-05-13 16:15:36,689] A new study created in memory with name: legalbert-v2-hpo



Optuna HPO: 10 trials | Metric: Val Macro F1 (maximize)



  0%|          | 0/10 [00:00<?, ?it/s]


TRIAL 00 | lr_enc=4.3e-06 | lr_head=4.1e-03 | dropout=0.4 | wd=0.06
         bs=4 | accum=2 | freeze=0 | γ_neg=6 | γ_pos=1 | clip=0.025


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1/6 [FROZEN] | Loss: 0.2168 | Macro F1: 0.1675
  Epoch  2/6 [FULL  ] | Loss: 0.1620 | Macro F1: 0.2227
  Epoch  3/6 [FULL  ] | Loss: 0.1478 | Macro F1: 0.2051
  Epoch  4/6 [FULL  ] | Loss: 0.1409 | Macro F1: 0.2165
  → Pruned at epoch 4
[I 2026-05-13 20:26:08,978] Trial 8 pruned. 

TRIAL 09 | lr_enc=2.5e-05 | lr_head=2.9e-03 | dropout=0.1 | wd=0.05
         bs=4 | accum=4 | freeze=2 | γ_neg=5 | γ_pos=0 | clip=0.100


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1/6 [FROZEN] | Loss: 0.1018 | Macro F1: 0.3142
  Epoch  2/6 [FROZEN] | Loss: 0.0926 | Macro F1: 0.3446
  Epoch  3/6 [FULL  ] | Loss: 0.0893 | Macro F1: 0.3299
  Epoch  4/6 [FULL  ] | Loss: 0.0857 | Macro F1: 0.3368
  Epoch  5/6 [FULL  ] | Loss: 0.0727 | Macro F1: 0.3365
  → Early stop (patience=3)
  Trial 09 best Macro F1 = 0.3446
[I 2026-05-13 20:51:38,575] Trial 9 finished with value: 0.34455855447620404 and parameters: {'lr_encoder': 2.453480159918337e-05, 'lr_head': 0.0028997158521665897, 'dropout': 0.1, 'weight_decay': 0.05, 'batch_size': 4, 'grad_accum': 4, 'freeze_epochs': 2, 'gamma_neg': 5, 'gamma_pos': 0, 'clip': 0.1}. Best is trial 2 with value: 0.39669460373148274.

BEST TRIAL
Best Macro F1: 0.3967
Hyperparameters:
  lr_encoder: 1.015066704592857e-05
  lr_head: 0.00011992724522955161
  dropout: 0.35
  weight_decay: 0.01
  batch_size: 16
  grad_accum: 2
  freeze_epochs: 3
  gamma_neg: 4
  gamma_pos: 0
  clip: 0.05

Results saved: /kaggle/working/bert_final/optuna_res

## 11. Final Training với Best Params

In [13]:
print("\n" + "=" * 65)
print("FINAL TRAINING")
print("=" * 65)
 
p = best.params
 
final_train_loader = DataLoader(train_dataset, batch_size=p["batch_size"],
                                shuffle=True,  num_workers=2)
final_val_loader   = DataLoader(val_dataset,   batch_size=p["batch_size"],
                                shuffle=False, num_workers=2)
 
final_model = ManipulationClassifier(
    BASE_MODEL, NUM_LABELS,
    dropout=p["dropout"],
    k_layers=K_LAYERS,
    gamma_pos=p["gamma_pos"],
    gamma_neg=p["gamma_neg"],
    clip=p["clip"]
).to(DEVICE)
 
freeze_ep = p["freeze_epochs"]
if freeze_ep > 0:
    final_model.freeze_encoder()
    optimizer_final = AdamW(
        filter(lambda x: x.requires_grad, final_model.parameters()),
        lr=p["lr_head"], weight_decay=p["weight_decay"]
    )
else:
    optimizer_final = AdamW([
        {'params': final_model.encoder.parameters(),
         'lr': p["lr_encoder"], 'weight_decay': p["weight_decay"]},
        {'params': final_model.classifier.parameters(),
         'lr': p["lr_head"],    'weight_decay': p["weight_decay"]}
    ])
 
total_steps_final = len(final_train_loader) * EPOCHS_FINAL
scheduler_final   = get_linear_schedule_with_warmup(
    optimizer_final,
    num_warmup_steps=total_steps_final // 10,
    num_training_steps=total_steps_final
)
 
best_f1_final    = 0.0
best_thresholds  = np.full(NUM_LABELS, 0.5)
patience_counter = 0
 
for epoch in range(EPOCHS_FINAL):
    if epoch == freeze_ep and freeze_ep > 0:
        print(f"\nUnfreezing encoder at epoch {epoch + 1}...")
        final_model.unfreeze_encoder()
        optimizer_final = AdamW([
            {'params': final_model.encoder.parameters(),
             'lr': p["lr_encoder"], 'weight_decay': p["weight_decay"]},
            {'params': final_model.classifier.parameters(),
             'lr': p["lr_head"],    'weight_decay': p["weight_decay"]}
        ])
        remaining = EPOCHS_FINAL - freeze_ep
        scheduler_final = get_linear_schedule_with_warmup(
            optimizer_final,
            num_warmup_steps=len(final_train_loader) * remaining // 10,
            num_training_steps=len(final_train_loader) * remaining
        )
        patience_counter = 0
 
    final_model.train()
    total_loss = 0.0
    optimizer_final.zero_grad()
 
    for step, batch in enumerate(final_train_loader):
        iids   = batch['input_ids'].to(DEVICE)
        amask  = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
 
        loss, _ = final_model(iids, amask, labels)
        (loss / p["grad_accum"]).backward()
 
        if (step + 1) % p["grad_accum"] == 0 or (step + 1) == len(final_train_loader):
            nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
            optimizer_final.step()
            scheduler_final.step()
            optimizer_final.zero_grad()
 
        total_loss += loss.item()
 
    avg_loss = total_loss / len(final_train_loader)
    macro_f1, val_probs, val_labels_arr = evaluate_macro_f1(
        final_model, final_val_loader,
        val_dataset.chunk_to_sample, len(val_texts)
    )
 
    phase = "[FROZEN]" if epoch < freeze_ep else "[FULL  ]"
    print(f"Epoch {epoch+1:>2}/{EPOCHS_FINAL} {phase} | "
          f"Loss: {avg_loss:.4f} | Macro F1: {macro_f1:.4f}")
 
    if macro_f1 > best_f1_final:
        best_f1_final   = macro_f1
        best_thresholds = tune_thresholds(val_probs, val_labels_arr)
        patience_counter = 0
        torch.save(final_model.state_dict(), f"{OUTPUT_DIR}/best_model.pt")
        np.save(f"{OUTPUT_DIR}/best_thresholds.npy", best_thresholds)
        print(f"  ✓ Saved (Macro F1: {best_f1_final:.4f})")
    else:
        patience_counter += 1
        if epoch == freeze_ep:
            patience_counter = 0
        if epoch >= freeze_ep and patience_counter >= PATIENCE_FINAL:
            print(f"\nEarly stopping at epoch {epoch + 1}")
            break
 
print(f"\nFinal Best Val Macro F1: {best_f1_final:.4f}")
print(f"Model: {OUTPUT_DIR}/best_model.pt")
print(f"Thresholds: {OUTPUT_DIR}/best_thresholds.npy")


FINAL TRAINING


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch  1/30 [FROZEN] | Loss: 0.1663 | Macro F1: 0.1972

Technique                   Thresh       F1        P        R
-------------------------------------------------------
  persuasion                 0.375    0.226    0.128    1.000
  playing the victim         0.425    0.282    0.168    0.875
  gaslighting                0.475    0.516    0.369    0.857
  evasion                    0.475    0.358    0.232    0.792
  deflection                 0.625    0.623    0.585    0.667
  minimization               0.475    0.549    0.418    0.800
  dismissal                  0.300    0.070    0.036    1.000
  guilt tripping             0.325    0.138    0.083    0.400
  emotional appeal           0.375    0.298    0.175    1.000
  framing the narrative      0.300    0.147    0.083    0.625
  character attack           0.250    0.051    0.028    0.333
  ✓ Saved (Macro F1: 0.1972)
Epoch  2/30 [FROZEN] | Loss: 0.1385 | Macro F1: 0.2749

Technique                   Thresh       F1        P       